# Synthetic Data Generation for Rationale Equivalence

This notebook documents the full pipeline for generating synthetic training data used to train a **rationale equivalence classifier**.

The classifier's task is to determine whether two natural-language rationales convey the **same meaning** (equivalent) or a **different meaning** (unequivalent). This kind of labelled pair data is expensive to collect from humans at scale, so it is generated synthetically using a large language model.

---

## What problem does this solve?

The downstream model (a fine-tuned RoBERTa classifier) needs to learn to distinguish between:
- **Equivalent pairs**: two texts that express the same underlying claim or verdict.
- **Unequivalent pairs**: two texts where the meaning has changed

Rather than hand-labelling thousands of pairs, we use a capable LLM (Qwen 2.5) to produce the synthetic counterpart for each original rationale. The label (`equivalent` or `unequivalent`) is controlled deterministically by which prompt we give the model.

---

## Two iterations of the pipeline

| Version | Approach | Model | Key idea |
|---------|----------|-------|----------|
| **v1 – Original** | Zero-shot | Qwen 2.5-3B-Instruct-GPTQ-Int4 | Static system prompt, no examples |
| **v2 – Improved** | Few-shot | Qwen 2.5-3B-Instruct-GPTQ-Int4 | Dynamic prompts seeded with manually labelled human examples |

Both versions use the same model. The difference is entirely in the prompting strategy.

---
## 0. Imports and dependencies

In [1]:
import argparse
import random

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# vLLM is a high-throughput inference engine for large language models.
# It handles batching, KV-cache management, and GPU memory allocation
# efficiently, which makes it practical to run generation over thousands
# of rows in a reasonable amount of time.
from vllm import LLM, SamplingParams

---
## 1. Source dataset

The input is a CSV file derived from the **HateCOT** dataset a collection of social media posts annotated for hate speech, each accompanied by a natural-language explanation (rationale) of why the post does or does not constitute hate speech.

Each row contains:
- `post` — the original social media text
- `explanation` — the rationale for the hate speech classification
- `label` — the hate speech verdict (`hate` / `not hate`)
- `domain` — which sub-dataset the example comes from (used later for domain-OOD evaluation)

The `explanation` column is the field we will be rewriting synthetically.

In [2]:
# Path to the curated source file. Change this to point to your local copy.
INPUT_PATH = "hatecot_final_D3.csv"

# The column whose text we will rewrite. Every other column is preserved
# unchanged in the output, so the synthetic file remains aligned with the
# original metadata (post, label, domain, etc.).
TARGET_COLUMN = "explanation"

df = pd.read_csv(INPUT_PATH)

# Drop rows where the target column is empty — the model has nothing to
# rewrite and would produce degenerate outputs.
df = df[df[TARGET_COLUMN].notna()]
df = df[df[TARGET_COLUMN].astype(str).str.strip() != ""]

print(f"Total usable rows: {len(df)}")
print(df[["domain", "label"]].value_counts().sort_index())

Total usable rows: 52137
domain      label                     
anatomy     Hateful                       2925
            Neutral                       2493
cad         Affiliation Directed Abuse    1032
            Identity Directed Abuse       1245
            Neutral                       3591
            Person Directed Abuse          849
dynahate    Animosity                     1326
            Dehumanization                 513
            Derogation                    1293
            Not Hate                      4077
gab         Hate                          3219
            Not Hate                      1815
measure     Hate Speech                   3609
            Not Hate Speech               3663
socialbias  Not Offensive                 3006
            Offensive                     3027
toraman     Hate                          1767
            Normal                        3663
            Offensive                     1785
toxigen     Benign                        3

### Sampling

Full-dataset generation is run on a GPU cluster. For development and inspection we sample a small subset.

In [3]:
# Number of rows to process. Set to -1 to run the full dataset.
N_SAMPLES = 50
SEED = 42

if N_SAMPLES > 0:
    sample = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
else:
    sample = df.reset_index(drop=True)

print(f"Working with {len(sample)} rows.")
sample.head(3)

Working with 50 rows.


,id,explanation,domain,label,post,target,uid
0,1343468911942135809,The post does not contain any language that ta...,toraman,Normal,The incident itself could have been a watershe...,NaN,toraman_1343468911942135809
1,24394,The post should be classified as Hateful becau...,anatomy,Hateful,I hate these fake ass interviews.,NaN,anatomy_24394
2,5276,The post targets both men and women by discuss...,measure,Not Hate Speech,Women should tell the truth. Like the sex got ...,"Men , Women",measure_5276


### Label assignment

Each row in the output will carry one of two labels:
- **`equivalent`** — the synthetic text should preserve the meaning of the original.
- **`unequivalent`** — the synthetic text should change the meaning in a detectable way.

Labels are assigned by simple alternation (even index → equivalent, odd index → unequivalent). This guarantees a **balanced 50 / 50 class split** regardless of dataset size, which is important for training a binary classifier without needing to re-weight classes.

In [4]:
# Alternating assignment produces an exactly balanced dataset.
labels = [
    "equivalent" if i % 2 == 0 else "unequivalent"
    for i in range(len(sample))
]

print(f"equivalent:   {labels.count('equivalent')}")
print(f"unequivalent: {labels.count('unequivalent')}")

equivalent:   25
unequivalent: 25


---
## 2. Model setup

The model is loaded **once** and reused across both generation pipelines. Both pipelines use identical inference settings — only the prompts differ.

**Model:** `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` (4-bit GPTQ, ~4 GB VRAM)

In [5]:
# dtype=float16 is required on NVIDIA V100 GPUs, which do not support
# bfloat16 hardware instructions.
llm = LLM(
    model="Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    tensor_parallel_size=1,
    max_model_len=2048,
    gpu_memory_utilization=0.85,
    dtype="float16",
    quantization="gptq",
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()

# Shared sampling parameters used by both pipelines.
sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=512)

INFO 06-05 13:44:51 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'gptq', 'model': 'Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4'}
INFO 06-05 13:44:52 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 06-05 13:44:52 [model.py:1678] Using max model len 2048
INFO 06-05 13:44:56 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 06-05 13:44:56 [gptq.py:99] Currently, the 4-bit gptq_gemm kernel for GPTQ is buggy. Please switch to gptq_marlin.
INFO 06-05 13:44:57 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=3192) INFO 06-05 13:44:58 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trus

(EngineCore pid=3192) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=3192) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(EngineCore pid=3192) INFO 06-05 13:45:02 [weight_utils.py:625] No model.safetensors.index.json found in remote.
(EngineCore pid=3192) INFO 06-05 13:45:02 [weight_utils.py:848] Prefetching checkpoint files into page cache started (in background)


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore pid=3192) INFO 06-05 13:45:03 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/1)
(EngineCore pid=3192) INFO 06-05 13:45:03 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 0.51s
(EngineCore pid=3192) INFO 06-05 13:45:03 [default_loader.py:384] Loading weights took 0.71 seconds
(EngineCore pid=3192) INFO 06-05 13:45:04 [gpu_model_runner.py:4820] Model loading took 1.95 GiB memory and 3.058032 seconds
(EngineCore pid=3192) INFO 06-05 13:45:12 [backends.py:1051] Using cache directory: /home/.cache/vllm/torch_compile_cache/1e2d2c7512/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=3192) INFO 06-05 13:45:12 [backends.py:1111] Dynamo bytecode transform time: 7.68 s
(EngineCore pid=3192) INFO 06-05 13:45:18 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 4.473 s
(EngineCore pid=3192) INFO 06-05 13:45:18 [decorators.py:303] Directly load AOT compilation from path /home/.cache/

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 11.52it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 12.58it/s]


(EngineCore pid=3192) INFO 06-05 13:45:31 [gpu_model_runner.py:6046] Graph capturing finished in 8 secs, took 0.72 GiB
(EngineCore pid=3192) INFO 06-05 13:45:31 [gpu_worker.py:597] CUDA graph pool memory: 0.72 GiB (actual), 0.74 GiB (estimated), difference: 0.03 GiB (3.8%).
(EngineCore pid=3192) INFO 06-05 13:45:31 [core.py:283] init engine (profile, create kv cache, warmup model) took 26.81 seconds


---
## 2. Pipeline v1 — Zero-Shot Reformulation

The first version of the pipeline instructs the model through **system prompts alone**, with no demonstrations. It rewrites hate speech rationales in two modes:

- **Equivalent** prompt: produce an exact paraphrase that preserves every claim, entity, and verdict.
- **Unequivalent** prompt: produce a rewrite where the core assertion is clearly different — typically by flipping a verdict, swapping an entity, or inverting a sentiment.

**Model:** `Qwen/Qwen2.5-3B-Instruct` (unquantized, ~6 GB VRAM)

In [6]:
# --- EQUIVALENT system prompt ---
# The key constraints are: preserve meaning, tone, named entities, numbers,
# and intent. The instruction to output ONLY the reformulated sentence prevents
# the model from adding preamble like "Sure, here is the paraphrase:" which
# would contaminate the output.
SYSTEM_PROMPT_EQUIVALENT = (
    "You are a precise paraphrasing assistant. Rewrite the user's sentence in "
    "different words while preserving the exact meaning, tone, named entities, "
    "numbers, and intent. Do not add, omit, or infer information. Output ONLY "
    "the reformulated sentence, with no preamble, no quotes, no commentary."
)

# --- UNEQUIVALENT system prompt ---
# The model is given a menu of legal transformations rather than a free brief.
# This specificity is necessary: without it, the model tends to produce
# near-paraphrases that are technically different but not detectably so —
# the resulting pairs would be mislabelled. The concrete example at the end
# anchors the model's behaviour in the zero-shot setting.
SYSTEM_PROMPT_UNEQUIVALENT = (
    "You are a sentence-rewriting assistant. Your job is to produce a sentence "
    "whose MEANING is clearly DIFFERENT from the original. The output must NOT "
    "be a paraphrase, a summary, or a shortened version of the input. Keep the "
    "topic, style, and approximate length similar, but apply at least ONE of "
    "the following transformations so the assertion changes:\n"
    "  - flip a verdict, claim, or negation (e.g. 'qualifies' -> 'does not "
    "qualify', 'is hate speech' -> 'is not hate speech', 'yes' -> 'no')\n"
    "  - swap a key entity, target group, person, or category for a different "
    "one\n"
    "  - invert the sentiment or stance (positive <-> negative, supports <-> "
    "opposes)\n"
    "  - change a key fact, number, or attribute\n"
    "Example:\n"
    "  Original: The post does not direct abuse at any identity and is a "
    "neutral statement.\n"
    "  Rewritten: The post directs abuse at a specific identity and is a "
    "hateful statement.\n"
    "A reader comparing the two sentences must immediately see they assert "
    "different things. Output ONLY the rewritten sentence, with no preamble, "
    "no quotes, no commentary."
)

In [7]:
def build_prompts(tokenizer, sentences: list[str], system_prompts: list[str]) -> list[str]:
    """
    Convert raw (sentence, system_prompt) pairs into the full prompt strings
    that vLLM will feed to the model.

    apply_chat_template inserts the model-specific special tokens
    (e.g. <|im_start|>, <|im_end|> for Qwen) and structures the
    system / user turns in the format the model was instruction-tuned on.
    Skipping this step would cause the model to ignore the system prompt
    or produce garbled output.
    """
    prompts = []
    for sentence, system in zip(sentences, system_prompts):
        messages = [
            {"role": "system", "content": system},
            {"role": "user",   "content": str(sentence)},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,  # appends the assistant turn opener
            )
        )
    return prompts

In [8]:
system_prompts_v1 = [
    SYSTEM_PROMPT_EQUIVALENT if label == "equivalent" else SYSTEM_PROMPT_UNEQUIVALENT
    for label in labels
]

originals_v1 = sample[TARGET_COLUMN].astype(str).tolist()
prompts_v1   = build_prompts(tokenizer, originals_v1, system_prompts_v1)

outputs_v1   = llm.generate(prompts_v1, sampling_params, use_tqdm=True)
synthetic_v1 = [output.outputs[0].text.strip() for output in outputs_v1]

Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [9]:
# Build the output dataframe by copying all original columns, then appending
# the two new columns. Keeping every original column is important so that
# downstream code (e.g. the domain-OOD split in judge_finetune) can still
# access fields like `domain` and `post`.
output_v1 = sample.copy()
output_v1["synthetic"] = synthetic_v1
output_v1["meaning"]   = labels

output_v1.to_csv("reformulated_v1.csv", index=False)
print(f"Saved {len(output_v1)} rows to reformulated_v1.csv")

# Quick sanity check: inspect a few pairs side by side.
output_v1[[TARGET_COLUMN, "synthetic", "meaning"]].head(6)

Saved 50 rows to reformulated_v1.csv


,explanation,synthetic,meaning
0,The post does not contain any language that ta...,The post does not include any language that di...,equivalent
1,The post should be classified as Hateful becau...,The post should be classified as neutral becau...,unequivalent
2,The post targets both men and women by discuss...,The post addresses men and women through discu...,equivalent
3,"The post ""No ragrats"" does not fall into any o...","The post ""No ragrats"" directs abuse towards a ...",unequivalent
4,This post does not contain any abusive speech ...,This posting lacks any abusive language direct...,equivalent
5,The post targets women by demeaning and object...,"The post, however, celebrates women by highlig...",unequivalent


---
## 3. Pipeline v2 — Few-Shot with Human Examples

The zero-shot pipeline produces grammatically correct rewrites, but they can feel mechanical and easy to distinguish from human-written text purely by style. For the classifier to generalise to real annotator comments, the training data should **resemble the distribution of human writing**.

Pipeline v2 addresses this by:
1. Shifting the task from *rewriting a rationale* to *writing a first-person human annotator comment*.
2. Seeding each system prompt with **3 randomly sampled examples** drawn from a small manually-labelled set of real human comments.

This few-shot approach nudges the model toward the informal, opinionated register that actual annotators use, rather than the neutral register of a paraphrase tool.

**Model:** `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` — same model as v1.

> The 7B model weights occupy ~7.5 GB, leaving no headroom for the KV cache on a 9.3 GB GPU. The key contribution of v2 is the **prompting strategy**, not the model size, so reusing the 3B model is the right trade-off here.

In [10]:
def load_fewshot_examples(manual_labels_path: str) -> dict:
    """
    Load the manually annotated CSV and split examples by label.

    The file has a header row followed by an additional descriptive row
    (skiprows=1 skips that second row so pandas reads only real data).
    Returns a dict with keys 'equivalent' and 'unequivalent', each holding
    a list of {post, human_comment} records used as few-shot demonstrations.
    """
    df = pd.read_csv(manual_labels_path, skiprows=1, header=0)

    # Normalise column names to the four fields we care about.
    df.columns = ["post", "explanation", "human_comment", "label"] + list(df.columns[4:])
    df = df[["post", "human_comment", "label"]].dropna(
        subset=["post", "human_comment", "label"]
    )

    # Normalise the label strings so capitalisation differences do not cause
    # examples to be silently dropped.
    df["label"] = df["label"].str.strip().str.lower()

    return {
        "equivalent":   df[df["label"] == "equivalent"][["post", "human_comment"]].to_dict("records"),
        "unequivalent": df[df["label"] == "unequivalent"][["post", "human_comment"]].to_dict("records"),
    }


fewshot = load_fewshot_examples("synthetic_data_generation_improved/manual_labelling.csv")

print(f"Few-shot pool — equivalent:   {len(fewshot['equivalent'])} examples")
print(f"Few-shot pool — unequivalent: {len(fewshot['unequivalent'])} examples")

Few-shot pool — equivalent:   13 examples
Few-shot pool — unequivalent: 7 examples


In [11]:
def _format_shots(examples: list[dict], n: int, seed: int) -> str:
    """
    Randomly sample n examples and format them as a numbered list
    that will be embedded directly in the system prompt.

    A fresh seed per row ensures that different rows see different
    subsets of the example pool, which reduces prompt repetition and
    makes the model less likely to copy-paste from the demonstrations.
    """
    rng = random.Random(seed)
    picked = rng.sample(examples, min(n, len(examples)))

    shots = ""
    for i, ex in enumerate(picked, 1):
        shots += (
            f"\nExample {i}:\n"
            f"  Post:    {ex['post'].strip()}\n"
            f"  Comment: {ex['human_comment'].strip()}\n"
        )
    return shots

In [12]:
def build_system_prompt_equivalent(examples: list[dict], n_shots: int = 3, seed: int = 0) -> str:
    """
    System prompt for the EQUIVALENT case.

    The model is asked to write a first-person comment that supports the AI
    label — i.e. a comment whose message is consistent with the original
    rationale. The instruction to avoid referencing the AI prevents the model
    from simply restating the label rather than reasoning about the post.
    """
    shots = _format_shots(examples, n_shots, seed)
    return (
        "You are a human annotator labelling social media posts for hate speech. "
        "You will be given a post and its AI classification. "
        "Write a short, casual, first-person comment stating YOUR OWN view of the post "
        "that supports the AI classification. "
        "Write as if you just read the post yourself - do NOT reference the AI, "
        "do NOT say 'I agree', just state your own view directly about the post. "
        "1-3 sentences, informal tone.\n\n"
        "Here are examples of how human annotators write their comments:"
        f"{shots}\n"
        "---\n"
        "Now write your own comment. Do NOT copy from the examples above. "
        "Output ONLY your comment, nothing else."
    )


def build_system_prompt_unequivalent(examples: list[dict], n_shots: int = 3, seed: int = 0) -> str:
    """
    System prompt for the UNEQUIVALENT case.

    Identical in structure to the equivalent prompt, but asks the model to
    write a comment that DISAGREES with the AI label. This generates a
    human-register text whose meaning is opposed to the original rationale.
    """
    shots = _format_shots(examples, n_shots, seed)
    return (
        "You are a human annotator labelling social media posts for hate speech. "
        "You will be given a post and its AI classification. "
        "Write a short, casual, first-person comment stating YOUR OWN view of the post "
        "that DISAGREES with the AI classification. "
        "Write as if you just read the post yourself - do NOT reference the AI, "
        "do NOT say 'I disagree', just state your own view directly about the post. "
        "1-3 sentences, informal tone.\n\n"
        "Here are examples of how human annotators write their comments:"
        f"{shots}\n"
        "---\n"
        "Now write your own comment. Do NOT copy from the examples above. "
        "Output ONLY your comment, nothing else."
    )

In [13]:
# For v2, the user message contains both the post text and the AI label,
# because the model needs to know what verdict to support or oppose.
# Concatenating them into a single user turn is simpler than using a
# multi-turn dialogue and produces equally good results in practice.
originals_v2 = (
    "Post: " + sample["post"].astype(str) +
    "\nAI classification: " + sample["label"].astype(str)
).tolist()

# Build one system prompt per row. The seed is set to the row index so
# each row gets a reproducible but distinct few-shot selection.
system_prompts_v2 = [
    build_system_prompt_equivalent(fewshot["equivalent"], seed=i)
    if label == "equivalent"
    else build_system_prompt_unequivalent(fewshot["unequivalent"], seed=i)
    for i, label in enumerate(labels)
]

# Inspect the system prompt for the first row to verify formatting.
print("--- System prompt for row 0 ---")
print(system_prompts_v2[0])

--- System prompt for row 0 ---
You are a human annotator labelling social media posts for hate speech. You will be given a post and its AI classification. Write a short, casual, first-person comment stating YOUR OWN view of the post that supports the AI classification. Write as if you just read the post yourself - do NOT reference the AI, do NOT say 'I agree', just state your own view directly about the post. 1-3 sentences, informal tone.

Here are examples of how human annotators write their comments:
Example 1:
  Post:    Those communisits went to hell when the prohibition on selling them alcohol was struck down in 1969 (Drybones v. the Queen)
  Comment: I think that's an opinion not really an insult. While I don’t necessary like the cursing. I’d say that’s not hate.

Example 2:
  Post:    This gender war comedy has been dragged out for so long, I will be leaving those jokes in 2020. #healthylifestyle
  Comment: Without the context, it’s hard to say but I don’t like the fact that he

In [14]:
prompts_v2   = build_prompts(tokenizer, originals_v2, system_prompts_v2)
outputs_v2   = llm.generate(prompts_v2, sampling_params, use_tqdm=True)
synthetic_v2 = [output.outputs[0].text.strip() for output in outputs_v2]

Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/50 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [15]:
output_v2 = sample.copy()
output_v2["synthetic"] = synthetic_v2
output_v2["meaning"]   = labels

output_v2.to_csv("reformulated_v2.csv", index=False)
print(f"Saved {len(output_v2)} rows to reformulated_v2.csv")

output_v2[["post", TARGET_COLUMN, "synthetic", "meaning"]].head(6)

Saved 50 rows to reformulated_v2.csv


,post,explanation,synthetic,meaning
0,The incident itself could have been a watershe...,The post does not contain any language that ta...,I don’t really see anything negative or offens...,equivalent
1,I hate these fake ass interviews.,The post should be classified as Hateful becau...,"I understand where you're coming from, but cal...",unequivalent
2,Women should tell the truth. Like the sex got ...,The post targets both men and women by discuss...,I think this post is making a sexist point and...,equivalent
3,No ragrats,"The post ""No ragrats"" does not fall into any o...",I don't like how it's just labeled as a deroga...,unequivalent
4,Oh wow... Isn't he the busiest little bee. I c...,This post does not contain any abusive speech ...,There's a clear tone of disapproval and slight...,equivalent
5,"Agreed, i have kicked myself in the past for b...",The post targets women by demeaning and object...,"I disagree, honesty is important, even if it's...",unequivalent


---
## 4. Quality Validation

Generating data with a label does not guarantee the label is correct. Two complementary checks are used to validate the output:

| Check | What it measures | Expected outcome |
|-------|-----------------|------------------|
| **TF-IDF cosine similarity** | Surface-level lexical overlap between original and synthetic text | Equivalent pairs → high similarity; unequivalent pairs → lower similarity |
| **LLM flip-rate audit** | Whether unequivalent pairs actually change the verdict (semantic check) | ≥ 80% of unequivalent pairs should reach a detectably different conclusion |

Both checks complement each other: cosine similarity is fast but surface-level; the LLM audit is slower but semantically aware.

### 4.1 TF-IDF Cosine Similarity

A high cosine similarity between an equivalent pair means the rewrite shares most of the important vocabulary with the original a good sign that meaning is preserved. A low cosine similarity for an unequivalent pair indicates that the key terms changed, consistent with a meaning flip.

In [16]:
# Use v2 output for the quality checks; replace with output_v1 to audit
# the zero-shot pipeline instead.
df_check = output_v2.copy()

# TF-IDF vectors represent each text as a weighted bag of n-grams.
# Fitting the vectorizer on the union of both columns ensures both texts
# are projected into the same vocabulary space before comparing.
vectorizer = TfidfVectorizer()
all_texts  = df_check[TARGET_COLUMN].tolist() + df_check["synthetic"].tolist()
vectorizer.fit(all_texts)

originals_tfidf  = vectorizer.transform(df_check[TARGET_COLUMN])
synthetics_tfidf = vectorizer.transform(df_check["synthetic"])

# Compute the cosine similarity for each pair individually (diagonal of the
# full similarity matrix).
similarities = [
    cosine_similarity(originals_tfidf[i], synthetics_tfidf[i])[0][0]
    for i in range(len(df_check))
]
df_check["cosine_sim"] = similarities

# Report mean cosine similarity stratified by label.
print(df_check.groupby("meaning")["cosine_sim"].describe().round(3))

              count   mean    std    min    25%    50%    75%    max
meaning                                                             
equivalent     25.0  0.095  0.062  0.000  0.054  0.075  0.127  0.223
unequivalent   25.0  0.136  0.092  0.025  0.072  0.113  0.190  0.446


### 4.2 Semantic Flip-Rate Audit

Cosine similarity cannot detect meaning changes expressed through negation (`"is hate speech"` vs `"is not hate speech"` score almost identically). The flip-rate audit uses GPT-4o-mini as a semantic judge to answer: *do these two texts reach the same conclusion?*

A correctly generated unequivalent pair should cause the judge to respond `DIFFERENT`. If it responds `SAME`, the pair is mislabelled — the model failed to actually flip the meaning.

The target flip rate is **≥ 80%** — i.e. at least 8 out of 10 unequivalent pairs should be genuinely different in meaning.

In [17]:
import openai

# The audit is run on a random subset of unequivalent pairs to keep API
# costs manageable. 30 samples provides sufficient signal for a rough
# quality estimate.
AUDIT_N = 30
AUDIT_SEED = 0

unequivalent_rows = df_check[df_check["meaning"] == "unequivalent"].sample(
    n=min(AUDIT_N, len(df_check[df_check["meaning"] == "unequivalent"])),
    random_state=AUDIT_SEED,
)


def judge(original: str, synthetic: str, client: openai.OpenAI) -> str:
    """
    Ask GPT-4o-mini whether two texts convey the same or different meaning.

    Returns 'SAME', 'DIFFERENT', or 'ERROR'. The prompt is intentionally
    minimal to reduce the chance the model hedges — we want a binary verdict.
    """
    prompt = (
        "Do the following two sentences convey the SAME or DIFFERENT meaning?\n"
        f"A: {original}\n"
        f"B: {synthetic}\n"
        "Reply with exactly one word: SAME or DIFFERENT."
    )
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5,
            temperature=0.0,  # deterministic output
        )
        return response.choices[0].message.content.strip().upper()
    except Exception as e:
        return "ERROR"


# Initialise the OpenAI client (requires OPENAI_API_KEY in the environment).
client = openai.OpenAI()

verdicts = [
    judge(row[TARGET_COLUMN], row["synthetic"], client)
    for _, row in unequivalent_rows.iterrows()
]

n_different = verdicts.count("DIFFERENT")
n_same      = verdicts.count("SAME")
n_error     = verdicts.count("ERROR")
flip_rate   = n_different / (n_different + n_same) if (n_different + n_same) > 0 else 0

print(f"Audited {len(verdicts)} unequivalent pairs")
print(f"  DIFFERENT (correct flip): {n_different}")
print(f"  SAME      (failed flip):  {n_same}")
print(f"  ERROR:                    {n_error}")
print(f"  Flip rate: {flip_rate:.1%}")

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

---
## 5. Summary and output schema

Both pipelines produce a CSV with the following columns:

| Column | Source | Description |
|--------|--------|-------------|
| `post` | original | The raw social media post |
| `explanation` | original | The human-written rationale |
| `label` | original | Hate speech verdict (`hate` / `not hate`) |
| `domain` | original | Sub-dataset identifier (used for domain-OOD splits) |
| `synthetic` | **generated** | The LLM-generated equivalent or unequivalent text |
| `meaning` | **generated** | `equivalent` or `unequivalent` |

This output is consumed directly by `judge_finetune/src/data.py`, which splits it by domain into train / val / test sets and tokenizes `(explanation, synthetic)` pairs for RoBERTa fine-tuning.

### Design decisions worth noting

- **Alternating label assignment** ensures a perfectly balanced 50/50 split without post-hoc resampling.
- **Per-row random seeds** for few-shot sampling prevent the model from seeing identical demonstrations across consecutive rows, which would reduce output diversity.
- **AWQ quantization** for the 7B model is a practical necessity on consumer-grade GPUs and introduces negligible quality degradation for this generation task.
- **The flip-rate audit** uses an LLM judge rather than rule-based heuristics because negation-based meaning changes (e.g. adding "not") are invisible to surface-level metrics like cosine similarity.

---
# Part II — Training the Judge, Diagnosing Its Failure, and Fixing It

Part I produced the synthetic `(explanation, synthetic, meaning)` pairs. The rest of this notebook documents what happens once those pairs are used to train the actual **rationale-equivalence judge**: how the RoBERTa model is fine-tuned, why its first results were *too good to be true*, what mechanistic interpretability revealed about the failure, and how a **multi-generator** training scheme repaired it.

> **Scope note.** The heavy experiments below were run as scripts on the GPU cluster — `roberta_training/`, `overnight/stage*.py`, and `mech_interp/`. The code in this part is **excerpted** from those scripts for readability; every reported number is read back from the stored result artifacts (`overnight/results/`, `verify_poster_results.json`, `mech_interp/results/`), not re-computed inline.

---
## 6. Fine-Tuning the RoBERTa Judge

The judge is **RoBERTa-base** (125M parameters, a Transformer encoder pre-trained with masked language modelling). We fine-tune it for a **binary sentence-pair** task: given two rationales, decide whether they are `equivalent` or `unequivalent`.

Each training example is the `(explanation, synthetic)` pair from Part I, encoded in RoBERTa's pair format:

```
<s> rationale_A </s></s> rationale_B </s>
```

The final-layer `<s>` representation is passed to a two-layer MLP classification head that emits two logits. The whole stack — encoder plus head — is fine-tuned end to end.

| Hyperparameter | Value |
|----------------|-------|
| Optimizer | AdamW |
| Learning rate | 2e-5 |
| Batch size | 32 |
| Epochs | 3 |
| Weight decay | 0.01 |
| Warmup | linear, 6% |
| Max length | 256 |
| Loss | cross-entropy |

**Data split.** The full synthetic set is 52,137 pairs across eight source domains. We split by **domain** so the test set is genuinely held out: we train on `{cad, socialbias, dynahate, toraman, measure}` (34,446 pairs) and evaluate on `{anatomy, gab, toxigen}` (17,691 pairs). Splitting by domain rather than at random is deliberate — it is the first, weakest form of out-of-distribution test.

In [ ]:
# Excerpt — roberta_training/run.py (the core fine-tuning setup)
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

args = TrainingArguments(
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    bf16=True,
)
# Trainer(model, args, train_dataset=..., eval_dataset=...).train()
# -> validation accuracy reaches ~0.99 within 2 epochs.

---
## 7. In-Distribution Evaluation — "Too Good to Be True"

We compare three judges on the held-out domains: a **TF-IDF + logistic-regression** baseline, **GPT-4o-mini zero-shot** (a real frontier LLM-as-a-Judge), and our **fine-tuned RoBERTa**.

| Model | Overall F1 | Anatomy | GAB | Toxigen |
|-------|-----------|---------|-----|---------|
| TF-IDF + LogReg | 0.789 | 0.759 | 0.776 | 0.815 |
| GPT-4o-mini zero-shot | 0.964 | 0.914 | 0.994 | 0.979 |
| RoBERTa fine-tuned (ours) | **0.968** | 0.949 | 0.976 | 0.977 |

These numbers are suspiciously good, and two observations explain why they should **not** be celebrated:

1. **A bag-of-words baseline already scores 0.789.** If TF-IDF — which sees only lexical overlap, no semantics — separates the classes this well, then lexical overlap alone is mostly sufficient. The task, as constructed, may not require semantic understanding at all.
2. **RoBERTa is statistically indistinguishable from GPT-4o-mini zero-shot** (0.968 vs 0.964). Beating a frontier model with a 125M encoder trained on synthetic data is a red flag, not a triumph.

A **cosine-similarity** analysis of the pairs confirms the suspicion:

| | Mean | Median | > 0.9 |
|---|------|--------|-------|
| Equivalent | 0.746 | 0.762 | 21.6% |
| Non-equivalent | 0.784 | 0.904 | 50.9% |

Non-equivalent pairs have a *higher* median similarity (0.904) than equivalent ones. They are not semantically distant — they differ by small, decisive word-level edits, most often a **negation flip**. A regex for negation predicts the label at AUC 0.72, which is exactly the shallow lexical cue we feared the model would latch onto.

In [ ]:
# Excerpt — verify_poster.py (TF-IDF baseline, domain-OOD split)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

Xtr = train["explanation"] + " [SEP] " + train["synthetic"]
Xte = test["explanation"]  + " [SEP] " + test["synthetic"]

vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(Xtr), train["label"])
preds = clf.predict(vec.transform(Xte))
# -> macro-F1 = 0.789 on the held-out domains (lexical overlap alone).

---
## 8. The Out-of-Distribution Collapse

The domain split above changes the *topic* but keeps the *writer* fixed — every rationale, train and test, is written by Qwen. The real test is a **different writer**: rationales hand-written by humans, or generated by a *different* LLM.

Here the model collapses. On a small set of human-written and Claude-written rationales it scores about **55%** — no better than chance — despite its 0.99 validation accuracy. It has not learned "do these two rationales mean the same thing"; it has learned "are these two rationales written in the same style as my training data". We call this the **semantic illusion**: a model that looks like a semantic judge on its own distribution but is really a style detector.

> This is the central failure the rest of the project investigates and fixes. The small evaluation sets (N = 20 human-written pairs) mean the exact number is noisy, but the *direction* — near-perfect in-distribution, near-random off-distribution — is unambiguous and reproducible.

---
## 9. Mechanistic Interpretability — Locating the Shortcut

To understand *where* and *how* the shortcut lives, we apply three interpretability methods to the trained encoder. (We deliberately restrict ourselves to methods we actually ran; we did **not** perform activation patching or integrated-gradient attribution.)

1. **Probing classifiers** — train a small logistic probe on the frozen `<s>` residual at each layer 0–12 and measure how decodable the equivalence label is.
2. **Directional ablation** — fit a linear direction for a suspected shortcut feature, project it out of the `<s>` residual, and re-measure the probe.
3. **Attention analysis** — inspect which token positions the model pools over when it decides.

### 9.1 Probing — the shortcut crystallises in the deep layers

We train probes on synthetic data (`val`) and test them on the human set (`manual`). The gap between the two tells the story:

| Layer | Probe AUC (val) | Probe AUC (human) |
|-------|-----------------|-------------------|
| 4 | 0.89 | 0.54 |
| 6 | 0.96 | 0.67 |
| 8 | 0.997 | 0.92 |
| 9 | 0.998 | **0.99** |
| 10 | 0.999 | 0.97 |
| 11 | 0.999 | 0.86 |
| 12 | 0.999 | 0.85 |

On synthetic data the equivalence signal is essentially perfect from layer 8 on. On human data the signal **peaks around layer 9 and then decays** through layers 10–12 — exactly the layers where the representation specialises to the training generator's writing style. The shortcut crystallises in the final third of the network.

In [ ]:
# Excerpt — mech_interp/scripts/train_probes.py (per-layer linear probe)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

for layer in range(13):
    X_val = residuals[layer]["val"]      # <s> vector at this layer
    probe = LogisticRegression(max_iter=1000).fit(X_val, y_val)
    auc_val    = roc_auc_score(y_val,    probe.predict_proba(X_val)[:, 1])
    auc_human  = roc_auc_score(y_human,  probe.predict_proba(X_human)[:, 1])
    # val saturates by layer 8; human AUC peaks ~layer 9 then decays.

### 9.2 Directional ablation — the shortcut is redundantly encoded

If the shortcut were a single direction in activation space, projecting that direction out of the `<s>` residual would destroy it. We try this at layer 12, removing up to six hand-picked stylistic directions (negation flip, length ratio, "the post" prefix, first-person, much-shorter, verdict word).

| Directions removed (cumulative) | Human-eval AUC |
|---|---|
| 0 (none) | 0.769 |
| +negation flip | 0.791 |
| +length ratio | 0.769 |
| +"the post" prefix | 0.780 |
| +first-person | 0.758 |
| +much-shorter | 0.769 |
| +verdict word | 0.758 |

Removing all six directions moves the AUC by **0.011** — essentially nothing. The shortcut is not localised in one direction; it is **redundantly encoded** across many. This matches the amnesic-probing literature (Elazar et al., 2021): a property smeared across many directions cannot be surgically removed at inference. The implication is decisive — *the shortcut has to be prevented during training, not erased afterwards.*

### 9.3 Attention — a single decisive position

The attention analysis adds the final piece: the classifier concentrates its decision on a **single token position** whose content reflects the training-data writing style, rather than integrating evidence across the rationale. A judge that reads one stylistic position is exactly what produces high in-distribution scores and an off-distribution collapse.

---
## 10. The Fix — Multi-Generator Training

If the shortcut is a *generator-specific style*, the cure is **generator diversity**. We regenerate the pairs from the **same 100 held-out, out-of-domain source posts** using five instruction-tuned LLMs — Claude Haiku 4.5, GPT-4o-mini, Qwen2.5-72B, Llama 3.1 8B, and DeepSeek V3 — at 200 pairs each (100 equivalent, 100 non-equivalent), for **~1,000 pairs total**. We then fine-tune a fresh RoBERTa on the union. (A sixth generator, Mistral-7B, silently produced no usable rows and is excluded.)

The result, evaluated across human-written and cross-LLM rationales:

| Training recipe | Handwritten (n=20) | Claude 4.6 (n=20) | GPT-4o-mini (n=50) | Claude Haiku (n=100) |
|---|---|---|---|---|
| Single-generator | 55% | 65% | 54% | 59% |
| **Multi-generator** | **95%** | **85%** | **72%** | **81%** |
| Drop Claude Haiku | 60% | 55% | 42% | 60% |
| Drop DeepSeek | 95% | 85% | 76% | 80% |

Two findings stand out:

- **Diversity beats scale.** ~1,000 *diverse* pairs eliminate the collapse, whereas 52,000 *single-generator* pairs did not. And in a leave-one-out test at a fixed ~800-pair budget, *which* generators you keep swings accuracy from 60% (drop Claude Haiku) to 95% (drop DeepSeek) — so it is the composition, not the count, that matters.
- **Generators are not interchangeable.** Claude Haiku carries unique signal (dropping it is catastrophic); DeepSeek is redundant (dropping it changes nothing).

In [ ]:
# Excerpt — overnight/stage9_matrix.py (build the multi-generator union)
import pandas as pd

slugs = ["anthropic_haiku", "openai_gpt4omini", "qwen25_72b",
         "meta_llama3_8b", "deepseek_v3"]          # mistral_7b produced 0 rows
union = pd.concat([load_generator_csv(s) for s in slugs], ignore_index=True)
print(len(union))                                   # -> 998 pairs (~1,000)
# train_classifier(union, "multi_gen", epochs=3)

---
## 11. Conclusions

- Training a small encoder on single-generator synthetic data produces a **semantic illusion**: near-perfect validation scores hiding a stylistic shortcut that collapses on any rationale written by a different hand.
- Mechanistic interpretability localises the shortcut to the **deep layers** and shows it is **redundantly encoded**, so it cannot be ablated away at inference — it must be prevented during training.
- **Generator diversity** is that prevention. ~1,000 cross-generator pairs reach **72–95%** on human and synthetic data and recover roughly 80% of a frontier LLM's judging accuracy at a fraction of the cost, decisively outperforming raw data scale.
- The remaining honest limitation is the task framing itself: forcing subjective human judgements about hate speech into a strict binary `equivalent` / `unequivalent` label is a real ceiling on how well any judge can do.